# 04 — Round-Trip: Code → GNN → Code

This notebook wires notebooks 01–03 together into a single **round-trip**:

```
    calculator/ (Python)
        ↓  forward scan
    model.gnn.md
        ↓  reverse synthesize
    calculator_reconstructed/ (Python)
        ↓  forward scan again
    model_reconstructed.gnn.md
```

A perfect round-trip is unattainable — GNN is a lossy semantic projection of code — but the **role distribution, state-space cardinalities, and matrix shapes** should line up. We use `cogant.reverse.metrics.compute_isomorphism_report` to measure that.

Run from the `cogant/` directory:
```bash
uv run jupyter nbconvert --to notebook --execute docs/notebooks/04_roundtrip.ipynb
```

## 1. Forward: scan the calculator fixture → GNN

In [1]:
from __future__ import annotations

import tempfile
from collections import Counter
from pathlib import Path

from cogant.api.session import Session

CANDIDATE_FIXTURES = [
    Path("tests/fixtures/calculator/"),
    Path("examples/control_positive/calculator/"),
]
FIXTURE = next((p.resolve() for p in CANDIDATE_FIXTURES if p.exists()), None)
if FIXTURE is None:
    raise FileNotFoundError("No calculator fixture found; run from cogant/.")
print(f"Fixture: {FIXTURE}")

Fixture: /Users/4d/Documents/GitHub/template/projects_in_progress/cogant/cogant/examples/control_positive/calculator


In [2]:
workdir = Path(tempfile.mkdtemp(prefix="cogant_nb04_"))
forward_out = workdir / "forward"

original_session = Session(repo_path=FIXTURE)
original_session.export_all(str(forward_out))

original_gnn_paths = sorted(forward_out.rglob("model.gnn.md"))
if not original_gnn_paths:
    raise FileNotFoundError(f"No model.gnn.md produced under {forward_out}")
original_gnn_path = original_gnn_paths[0]
print(f"Original GNN written: {original_gnn_path}")

Original GNN written: /var/folders/vc/rgmbpjpj0dbg61vr54xjskc80000gn/T/cogant_nb04_0hlro2fq/forward/gnn_package/model.gnn.md


## 2. Reverse: parse GNN → plan → synthesize

In [3]:
from cogant.reverse import parse_gnn, plan_package, synthesize_package

original_model = parse_gnn(original_gnn_path)
plan = plan_package(original_model)

synth_parent = workdir / "reverse"
synth_parent.mkdir(exist_ok=True)
package_path = synthesize_package(plan, original_model, synth_parent)

print(f"Synthesised package: {package_path}")
print("Package contents:")
for p in sorted(package_path.rglob("*.py")):
    print(f"  {p.relative_to(package_path)}")

Synthesised package: /private/var/folders/vc/rgmbpjpj0dbg61vr54xjskc80000gn/T/cogant_nb04_0hlro2fq/reverse/calculator
Package contents:
  __init__.py
  act.py
  constraints.py
  context.py
  main.py
  matrices.py
  observe.py
  policy.py
  state.py
  tests/__init__.py
  tests/test_smoke.py


## 3. Forward again: scan the reconstructed package → GNN

This is the second leg of the round-trip. We re-run the full COGANT forward pipeline on the synthesised package and obtain a fresh GNN file, which we can then compare against the original.

In [4]:
reverse_out = workdir / "forward2"
reconstructed_session = Session(repo_path=package_path)
reconstructed_session.export_all(str(reverse_out))

reconstructed_gnn_paths = sorted(reverse_out.rglob("model.gnn.md"))
if not reconstructed_gnn_paths:
    raise FileNotFoundError(f"No reconstructed model.gnn.md under {reverse_out}")
reconstructed_gnn_path = reconstructed_gnn_paths[0]
print(f"Reconstructed GNN: {reconstructed_gnn_path}")

reconstructed_model = parse_gnn(reconstructed_gnn_path)
print(f"Reconstructed hidden states: {reconstructed_model.hidden_states}")
print(f"Reconstructed observations : {reconstructed_model.observations}")
print(f"Reconstructed actions      : {reconstructed_model.actions}")

Reconstructed GNN: /var/folders/vc/rgmbpjpj0dbg61vr54xjskc80000gn/T/cogant_nb04_0hlro2fq/forward2/gnn_package/model.gnn.md
Reconstructed hidden states: ['s_f0']
Reconstructed observations : ['o_m0', 'o_m1', 'o_m2', 'o_m3', 'o_m4', 'o_m5', 'o_m6', 'o_m7', 'o_m8']
Reconstructed actions      : ['u_c0', 'u_c1', 'u_c2', 'u_c3', 'u_c4', 'u_c5', 'u_c6', 'u_c7', 'u_c8', 'u_c9', 'u_c10']


## 4. Role multisets and the isomorphism report

We build a role-count dict for each model from `ActInfOntologyAnnotation` and hand both to `compute_isomorphism_report`. The report returns an overall `total_score` in `[0, 1]` and a boolean `is_isomorphic` under the default threshold (0.7).

In [5]:
def roles_from(model) -> dict[str, int]:
    """Count the ActInfOntologyAnnotation role labels."""
    return dict(Counter(model.annotations.values()))

def matrices_from(model) -> dict[str, object]:
    return {
        "A": model.A,
        "B": model.B,
        "C": model.C,
        "D": model.D,
    }

roles_original = roles_from(original_model)
roles_reconstructed = roles_from(reconstructed_model)

print("Original roles      :", roles_original)
print("Reconstructed roles :", roles_reconstructed)

Original roles      : {'HiddenState': 1, 'PriorBelief': 1, 'TransitionMatrix': 1, 'LikelihoodMatrix': 1, 'Observation': 3, 'PreferenceVector': 3, 'Action': 6, 'ExpectedFreeEnergy': 1, 'Time': 1}
Reconstructed roles : {'HiddenState': 1, 'PriorBelief': 1, 'TransitionMatrix': 1, 'LikelihoodMatrix': 1, 'Observation': 9, 'PreferenceVector': 9, 'Action': 11, 'ExpectedFreeEnergy': 1, 'Time': 1}


In [6]:
from cogant.reverse.metrics import (
    compute_isomorphism_report,
    compare_role_distributions,
)

gnn_a = {"roles": roles_original, "matrices": matrices_from(original_model)}
gnn_b = {"roles": roles_reconstructed, "matrices": matrices_from(reconstructed_model)}

report = compute_isomorphism_report(gnn_a, gnn_b)

# Also expose the role match score directly.
role_match_score = compare_role_distributions(roles_original, roles_reconstructed)

print(report.summary())
print(f"role_match_score   : {role_match_score:.3f}")
print(f"total_score        : {report.total_score:.3f}")
print(f"is_isomorphic      : {report.is_isomorphic}")
print(f"breakdown          : {report.breakdown}")

[ISO] total=0.90 role=0.97 matrix=0.77 struct=1.00
role_match_score   : 0.967
total_score        : 0.896
is_isomorphic      : True
breakdown          : {'role_score': 0.9666915942478221, 'matrix_score': 0.7736418318465836, 'structural_score': 1.0, 'weights': {'role': 0.4, 'matrix': 0.4, 'structural': 0.2}, 'threshold': 0.7, 'roles_a_total': 18.0, 'roles_b_total': 35.0, 'matrix_keys_a': ['A', 'B', 'C', 'D'], 'matrix_keys_b': ['A', 'B', 'C', 'D'], 'n_nodes_a': 0, 'n_nodes_b': 0, 'n_edges_a': 0, 'n_edges_b': 0, 'per_matrix_frobenius': {'A': 0.5176380902049321, 'B': 0.38779458240873316, 'C': 0.0, 'D': 0.0}}


### Classify the outcome

We convert the continuous total score into one of three labels:

- **ISOMORPHIC** — `total_score >= 0.90` (round-trip recovered the model).
- **APPROXIMATE** — `0.70 <= total_score < 0.90` (minor drift, still acceptable).
- **DIVERGENT** — `total_score < 0.70` (something is wrong; investigate).

The value `ε = 1.0 - total_score` is the gap to a perfect round-trip.

In [7]:
epsilon = 1.0 - report.total_score
if report.total_score >= 0.90:
    classification = "ISOMORPHIC"
elif report.total_score >= 0.70:
    classification = "APPROXIMATE"
else:
    classification = "DIVERGENT"

print(f"epsilon (1 - total) : {epsilon:.3f}")
print(f"classification      : {classification}")

epsilon (1 - total) : 0.104
classification      : APPROXIMATE


## 5. Visualise the role distributions side by side

In [8]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

all_roles = sorted(set(roles_original) | set(roles_reconstructed))
orig_counts = [roles_original.get(r, 0) for r in all_roles]
rec_counts = [roles_reconstructed.get(r, 0) for r in all_roles]

fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(all_roles))
width = 0.35
ax.bar([i - width / 2 for i in x], orig_counts, width=width, label="original")
ax.bar([i + width / 2 for i in x], rec_counts, width=width, label="reconstructed")
ax.set_xticks(list(x))
ax.set_xticklabels(all_roles, rotation=30, ha="right")
ax.set_ylabel("count")
ax.set_title(f"Role distribution — {classification} (ε={epsilon:.2f})")
ax.legend()
plt.tight_layout()
fig

<Figure size 800x400 with 1 Axes>

## 6. Takeaways

- Round-trip never guarantees source-level equality. It guarantees semantic/structural equality (roles, cardinalities, matrix shapes).
- `compute_isomorphism_report` uses a fixed weighting: `0.4 · role + 0.4 · matrix + 0.2 · structural`.
- Tracking `ε = 1 - total_score` over time is how COGANT's CI detects rule regressions. Any commit that lifts ε above the ISOMORPHIC threshold should explain why.